# 108 - The Full SPXW Chain, Live

[Open in Colab](https://colab.research.google.com/github/userFRM/ThetaDataDx/blob/main/notebooks/108_full_chain_stream.ipynb)

One notebook, one Python process, one streaming session, loaded as far as a Pro plan allows:

- quotes on the 15,000 SPXW contracts nearest the money, across every listed expiration (the Pro limit for option quote streams),
- every option trade in the market, from the full trade stream, which also sends the quotes around each trade,
- quotes on the 2,000 most-traded stocks and ETFs (the Pro limit for stock quote streams),
- every stock trade in the market, from the full trade stream.

It runs for five minutes and reports what came in: events per second, how far behind the notebook ran, CPU use, and how many events were dropped.

Each frame is decoded once, in Rust. The notebook reads the stream as Arrow record batches with `client.stream.batches()`, so Python handles a batch of rows at a time instead of one callback per event. There is no asyncio, no thread pool and no queue to manage here.

**Requires** a Pro plan on options and stocks during market hours, 09:30 - 16:00 ET: the full trade streams and the flat-file report used to rank stocks are Pro. On Standard, set `QUOTE_LIMIT` to 10,000 and `STOCK_QUOTES` to 1,000, pick the stocks yourself, and subscribe to per-contract trades as in [107](107_full_trade_stream.ipynb). One live session per account: a second session on the same account takes over the connection, so stop any other streaming client first.

In [ ]:
# In Colab or a fresh environment this installs the SDK first.
import importlib.util
if importlib.util.find_spec("thetadatadx") is None:
    %pip install -q "thetadatadx-py[all]" matplotlib

In [ ]:
import time
from collections import Counter, defaultdict
from datetime import datetime
from zoneinfo import ZoneInfo

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.compute as pc
from IPython.display import Markdown, display

from thetadatadx import Client, Config, Contract, Credentials, SecType

## 1. Connect

Credentials come from `THETADATA_API_KEY` or `creds.txt` when either is there. Otherwise, as in Colab, the cell asks for your email and password in hidden prompts, so neither is saved in the notebook. `wait_mode = "backoff"` keeps the event-ring consumer spinning while events flow and lets it sleep when the stream goes quiet, so an idle session does not hold a whole core.

In [ ]:
import os
from getpass import getpass

if os.environ.get("THETADATA_API_KEY") or os.path.exists("creds.txt"):
    creds = Credentials.from_env_or_file("creds.txt")
else:
    creds = Credentials(getpass("ThetaData email: "), getpass("ThetaData password: "))

config = Config.production()
config.wait_mode = "backoff"
client = Client(creds, config)
print(client)

## 2. The SPXW chain

One call lists every SPXW contract quoted today. SPX trades near ten times SPY, which is close enough to rank strikes by distance from the money; the quote streams go to the nearest ones first.

In [ ]:
QUOTE_LIMIT = 15_000  # option quote streams: 15,000 on Pro, 10,000 on Standard

today = datetime.now(ZoneInfo("America/New_York")).date()
spy = client.market_data.stock_snapshot_quote(["SPY"]).to_list()[0]
spot = 10 * (spy.bid + spy.ask) / 2

listed = client.market_data.option_list_contracts("quote", today, symbol="SPXW").to_list()
listed = [c for c in listed if c.expiration >= int(today.strftime("%Y%m%d"))]
listed.sort(key=lambda c: abs(c.strike - spot))
chain = [
    Contract.option("SPXW", expiration=str(c.expiration), strike=f"{c.strike:g}", right=c.right)
    for c in listed
]

print(f"SPX near {spot:,.0f}")
print(f"{len(chain):,} SPXW contracts across {len({c.expiration for c in listed})} expirations")
print(f"quotes on the nearest {len(chain[:QUOTE_LIMIT]):,}")

## 3. The most-traded stocks

The flat-file end-of-day report covers every stock in one call. Ranking it by the number of trades in the last session picks the names whose quotes move the most.

In [ ]:
STOCK_QUOTES = 2_000  # stock quote streams: 2,000 on Pro, 1,000 on Standard

for back in range(1, 6):  # the last session with a report
    eod = client.market_data.stock_flat_file_eod(today - pd.offsets.BDay(back)).to_pandas()
    if len(eod):
        break
stocks = eod.nlargest(STOCK_QUOTES, "count")["symbol"].tolist()

print(f"{len(eod):,} stocks in the report for {eod['date'].iloc[0]}")
print(f"quotes on the {len(stocks):,} most traded, starting with {', '.join(stocks[:10])}")

subscriptions = (
    [c.quote() for c in chain[:QUOTE_LIMIT]]
    + [Contract.stock(s).quote() for s in stocks]
    + [SecType.OPTION.full_trades(), SecType.STOCK.full_trades()]
)
print(f"{len(subscriptions):,} subscriptions")

## 4. Subscribe and run

`batches()` opens the session, so the reader comes first and the subscriptions after it. For every batch the notebook counts rows per stream and records the delay from the moment the SDK received the oldest row (`received_at_ns`) to the moment Python holds it. A batch is handed over when it fills or after `linger_ms`, so that delay includes up to 10 ms of waiting. The panel refreshes once a second.

In [ ]:
RUN_SECONDS = 300

per_second = defaultdict(Counter)   # wall-clock second -> rows per stream
delay_ms = defaultdict(float)       # wall-clock second -> worst receive-to-notebook delay
ring_peak = 0
panel = display(Markdown("Connecting..."), display_id=True)


def show(elapsed):
    total = sum(sum(c.values()) for c in per_second.values())
    last = per_second.get(int(time.time()) - 1, Counter())
    panel.update(Markdown(
        f"| running | events | last second | worst delay, last second | event ring peak | dropped |\n"
        f"|---|---|---|---|---|---|\n"
        f"| {elapsed:.0f} s | {total:,} | {sum(last.values()):,}/s "
        f"| {delay_ms.get(int(time.time()) - 1, 0):.1f} ms | {ring_peak:,} "
        f"| {client.stream.dropped_event_count():,} |"
    ))


cpu_start, start = time.process_time(), time.monotonic()
with client.stream.batches(batch_size=32_768, linger_ms=10) as batches:
    client.stream.subscribe_many(subscriptions)
    active = len(client.stream.active_subscriptions()) + len(client.stream.active_full_subscriptions())
    next_refresh = start + 1
    for batch in batches:
        second = int(time.time())
        asset = pc.if_else(pc.is_valid(batch["expiration"]), "option", "stock")  # only option rows carry an expiration
        stream = pc.binary_join_element_wise(asset, batch["event_type"], " ")
        for row in pc.value_counts(stream).to_pylist():
            per_second[second][row["values"]] += row["counts"]
        oldest = pc.min(batch["received_at_ns"]).as_py()
        delay_ms[second] = max(delay_ms[second], (time.time_ns() - oldest) / 1e6)
        ring_peak = max(ring_peak, client.stream.ring_occupancy())

        now = time.monotonic()
        if now >= next_refresh:
            show(now - start)
            next_refresh = now + 1
        if now - start >= RUN_SECONDS:
            break

    elapsed = time.monotonic() - start
    cpu_share = (time.process_time() - cpu_start) / elapsed
    dropped = client.stream.dropped_event_count()
    ring_size = client.stream.ring_capacity()
    show(elapsed)
print(f"Session closed after {elapsed:.0f} s.")

## 5. What came in

In [ ]:
rates = pd.DataFrame.from_dict(per_second, orient="index").fillna(0).sort_index()
delays = pd.Series(delay_ms).sort_index()
rates, delays = rates.iloc[1:-1], delays.iloc[1:-1]  # drop the partial first and last seconds
rates.index = delays.index = pd.to_datetime(rates.index, unit="s", utc=True).tz_convert("America/New_York")
total_rate = rates.sum(axis=1)

summary = pd.DataFrame({
    "events": rates.sum().astype(int),
    "mean per second": rates.mean().round(0).astype(int),
    "peak second": rates.max().astype(int),
}).sort_values("events", ascending=False)
summary.loc["all streams"] = [int(rates.values.sum()), int(total_rate.mean()), int(total_rate.max())]
display(summary)

print(f"active subscriptions       {active:,}")
print(f"run                        {rates.index[0]:%H:%M:%S} to {rates.index[-1]:%H:%M:%S} ET")
print(f"worst delay, median second {delays.median():.1f} ms")
print(f"worst delay, 99th pct      {delays.quantile(0.99):.1f} ms")
print(f"event ring peak            {ring_peak:,} of {ring_size:,} slots")
print(f"process CPU                {cpu_share:.0%} of one core")
print(f"dropped events             {dropped:,}")

In [ ]:
options = rates[[c for c in rates.columns if c.startswith("option")]].sum(axis=1)
stock = rates[[c for c in rates.columns if c.startswith("stock")]].sum(axis=1)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, height_ratios=[2, 1])
top.plot(options.index, options, color="#2a6fdb", linewidth=1.2, label="Options: SPXW quotes and every option trade")
top.plot(stock.index, stock, color="#e07b22", linewidth=1.2, label="Stocks: 2,000 quote streams and every stock trade")
top.set_ylabel("events per second")
top.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
top.set_title("Events per second, one Python process")
top.legend(frameon=False)
top.grid(alpha=0.3)

bottom.plot(delays.index, delays, color="#2a6fdb", linewidth=1.0)
bottom.set_ylabel("ms")
bottom.set_ylim(bottom=0)
bottom.set_title("Worst receive-to-notebook delay in each second")
bottom.grid(alpha=0.3)
bottom.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S", tz=rates.index.tz))
plt.tight_layout()
plt.show()

---

To react to each event as it arrives instead, register a callback with `client.streaming(on_event)`, as in [105](105_realtime_streaming.ipynb).

**Previous:** [107 - Full Trade Stream](107_full_trade_stream.ipynb)
**Back to start:** [101 - Getting Started](101_getting_started.ipynb)